# 01 — Running Python Code in Notebooks & Bash/tmux
**Task:** Sentiment Analysis  
**Goal:** Learn how to run Python interactively in Jupyter, execute shell commands, and manage long-running jobs with tmux.

---

## 1. Python Basics in a Notebook Cell

In [ ]:
# Standard Python — runs immediately in the kernel
text = "I absolutely love this product! It exceeded all my expectations."
words = text.lower().split()
print(f"Word count : {len(words)}")
print(f"Words      : {words}")

## 2. Magic Commands

In [ ]:
# %timeit — benchmark a cell expression
%timeit [w for w in text.lower().split() if len(w) > 3]

In [ ]:
%%time
# %%time — time the whole cell
import time
time.sleep(0.1)
print("Done sleeping")

In [ ]:
# %who — list variables in namespace
%who

In [ ]:
# %matplotlib inline — render plots inside the notebook
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

labels = ['Positive', 'Negative', 'Neutral']
counts = [450, 300, 250]
plt.bar(labels, counts, color=['#4CAF50', '#F44336', '#9E9E9E'])
plt.title('Sentiment Distribution (sample)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 3. Shell Commands with `!` and `%%bash`

In [ ]:
# Single shell command
!echo "Python version:"
!python --version
!pip show transformers | head -5

In [ ]:
# Install packages from inside the notebook
!pip install -q datasets transformers scikit-learn pandas nltk

In [ ]:
%%bash
# Multi-line bash cell
echo "--- System Info ---"
uname -a
echo ""
echo "--- GPU Info ---"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "No GPU found"
echo ""
echo "--- Disk Usage ---"
df -h /tmp

## 4. Capturing Shell Output into Python Variables

In [ ]:
# Capture output of a shell command
cpu_count = !nproc
print(f"CPU cores available: {cpu_count[0]}")

# Interpolate Python vars into shell commands
data_dir = "/tmp/sentiment_data"
!mkdir -p {data_dir} && echo "Directory created: {data_dir}"

## 5. Writing & Running a Python Script from the Notebook

In [ ]:
%%writefile /tmp/preprocess_simple.py
"""Minimal sentiment preprocessing script — run standalone."""
import re, sys

def clean(text: str) -> str:
    text = text.lower()
    text = re.sub(r'http\S+', '', text)       # remove URLs
    text = re.sub(r'[^a-z0-9\s]', '', text)  # keep alphanumeric
    return text.strip()

samples = [
    "I LOVE this! Check https://example.com 😊",
    "Terrible product, never buying again!!!",
    "It's okay, nothing special...",
]

for s in samples:
    print(f"IN : {s}")
    print(f"OUT: {clean(s)}")
    print()


In [ ]:
# Run the written script
!python /tmp/preprocess_simple.py

## 6. tmux — Managing Long-Running Jobs

Use `tmux` when training models that might take hours. You can detach and the job keeps running.

```bash
# --- From your terminal (not in the notebook) ---

# 1. Start a named tmux session
tmux new -s sentiment_train

# 2. Inside the tmux pane, run your script
python train_sentiment.py --epochs 10 --lr 2e-5 > logs/train.log 2>&1

# 3. Detach (leave running in background)
#    Press:  Ctrl+B  then  D

# 4. Re-attach later
tmux attach -t sentiment_train

# 5. List all sessions
tmux ls

# 6. Kill session when done
tmux kill-session -t sentiment_train
```

In [ ]:
# Check if tmux is available
!tmux -V 2>/dev/null || echo "tmux not installed — install with: sudo apt install tmux"

In [ ]:
%%bash
# Launch a background script via tmux non-interactively
# (useful for kicking off training from inside the notebook)
SCRIPT=/tmp/preprocess_simple.py
LOG=/tmp/sentiment_job.log

# Run in a detached tmux session
tmux new-session -d -s bg_job "python $SCRIPT > $LOG 2>&1" 2>/dev/null
sleep 1
echo "--- Job log ---"
cat $LOG 2>/dev/null || echo "Log not yet available"
tmux kill-session -t bg_job 2>/dev/null
echo "Done."

## 7. Useful Notebook Shortcuts

| Shortcut | Action |
|---|---|
| `Shift+Enter` | Run cell, move to next |
| `Ctrl+Enter` | Run cell, stay |
| `Esc → A` | Insert cell above |
| `Esc → B` | Insert cell below |
| `Esc → D D` | Delete cell |
| `Esc → M` | Change to Markdown |
| `Esc → Y` | Change to Code |
| `Ctrl+Shift+P` | Command palette (JupyterLab) |

---
**Next Notebook →** `02_preprocessing.ipynb`